In [0]:
# ============================================================
#  SILVER → CURRENT: KPIs
# ============================================================

STORAGE_ACCOUNT   = "stdatacolake"
CONTAINER_SILVER  = "silver-zone"
CONTAINER_CURRENT = "curated-zone"

# Rutas de lectura Silver
SILVER_VEHICULOS  = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/vehiculos"
SILVER_ENTREGAS   = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/entregas"
SILVER_BODEGAS    = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/bodegas"
SILVER_INVENTARIO = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/inventario"
SILVER_VENTAS     = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/ventas"
SILVER_VISITAS    = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/visitas"

# Rutas de escritura Current
CURRENT_KPI_ENTREGAS  = f"abfss://{CONTAINER_CURRENT}@{STORAGE_ACCOUNT}.dfs.core.windows.net/kpi_entregas_vehiculo"
CURRENT_KPI_INVENTARIO = f"abfss://{CONTAINER_CURRENT}@{STORAGE_ACCOUNT}.dfs.core.windows.net/kpi_inventario_bodega"
CURRENT_KPI_VENTAS    = f"abfss://{CONTAINER_CURRENT}@{STORAGE_ACCOUNT}.dfs.core.windows.net/kpi_ventas_vendedor"
CURRENT_KPI_VISITAS   = f"abfss://{CONTAINER_CURRENT}@{STORAGE_ACCOUNT}.dfs.core.windows.net/kpi_visitas_vendedor"

In [0]:
# ── LEER TABLAS SILVER ────────────────────────────────────────
df_vehiculos  = spark.read.format("delta").load(SILVER_VEHICULOS)
df_entregas   = spark.read.format("delta").load(SILVER_ENTREGAS)
df_bodegas    = spark.read.format("delta").load(SILVER_BODEGAS)
df_inventario = spark.read.format("delta").load(SILVER_INVENTARIO)
df_ventas     = spark.read.format("delta").load(SILVER_VENTAS)
df_visitas    = spark.read.format("delta").load(SILVER_VISITAS)

print("✅ Todas las tablas Silver cargadas")

In [0]:
# ── KPI 1: CUMPLIMIENTO Y EFICIENCIA POR VEHÍCULO ────────────
from pyspark.sql.functions import col, round, sum, avg, count, when, current_timestamp

df_kpi_entregas = df_entregas \
    .join(df_vehiculos, on="id_vehiculo", how="left") \
    .groupBy("id_vehiculo", "placa", "tipo", "conductor") \
    .agg(
        count("id_entrega").alias("total_entregas"),
        sum("cumplimiento").alias("entregas_cumplidas"),
        round(avg("tiempo_entrega_min"), 2).alias("tiempo_promedio_min"),
        round(sum("consumo_combustible"), 2).alias("consumo_total"),
        round(avg("eficiencia_combustible"), 2).alias("eficiencia_promedio")
    ) \
    .withColumn("pct_cumplimiento",
        round((col("entregas_cumplidas") / col("total_entregas")) * 100, 2)
    ) \
    .withColumn("semaforo_cumplimiento",
        when(col("pct_cumplimiento") >= 90, "VERDE")
       .when(col("pct_cumplimiento") >= 70, "AMARILLO")
       .otherwise("ROJO")
    ) \
    .withColumn("_updated_at", current_timestamp())

print(f"✅ KPI Entregas: {df_kpi_entregas.count()} vehículos")
df_kpi_entregas.show(5)

In [0]:
# ── KPI 2: STOCK Y ALERTAS POR BODEGA ────────────────────────
df_kpi_inventario = df_inventario \
    .join(df_bodegas, on="id_bodega", how="left") \
    .groupBy("id_bodega", "nombre_bodega", "departamento", "ciudad", "capacidad_max") \
    .agg(
        sum("stock_resultante").alias("stock_total"),
        count("id_movimiento").alias("total_movimientos"),
        sum(when(col("alerta_vencimiento") == "CRITICO", 1).otherwise(0)).alias("productos_criticos"),
        sum(when(col("alerta_vencimiento") == "PROXIMO", 1).otherwise(0)).alias("productos_proximos"),
        sum(when(col("tipo_movimiento") == "Entrada", col("cantidad")).otherwise(0)).alias("total_entradas"),
        sum(when(col("tipo_movimiento") == "Salida",  col("cantidad")).otherwise(0)).alias("total_salidas")
    ) \
    .withColumn("pct_ocupacion",
        round((col("stock_total") / col("capacidad_max")) * 100, 2)
    ) \
    .withColumn("semaforo_ocupacion",
        when(col("pct_ocupacion") >= 90, "CRITICO")
       .when(col("pct_ocupacion") >= 70, "ALTO")
       .otherwise("NORMAL")
    ) \
    .withColumn("_updated_at", current_timestamp())

print(f"✅ KPI Inventario: {df_kpi_inventario.count()} bodegas")
df_kpi_inventario.show(5)

In [0]:
df_ventas

In [0]:
# ── KPI 3: VENTAS POR VENDEDOR ────────────────────────────────
df_kpi_ventas = df_ventas \
    .groupBy("id_vendedor") \
    .agg(
        count("id_visita").alias("total_ventas"),
        sum(when(col("resultado") == "Acuerdo", 1).otherwise(0)).alias("ventas_ganadas"),
        round(sum("valor_cartera"), 2).alias("valor_cartera_total"),
        round(avg("valor_cartera"), 2).alias("valor_cartera_promedio")
    ) \
    .withColumn("pct_conversion",
        round((col("ventas_ganadas") / col("total_ventas")) * 100, 2)
    ) \
    .withColumn("semaforo_conversion",
        when(col("pct_conversion") >= 50, "VERDE")
       .when(col("pct_conversion") >= 30, "AMARILLO")
       .otherwise("ROJO")
    ) \
    .withColumn("_updated_at", current_timestamp())

print(f"✅ KPI Ventas: {df_kpi_ventas.count()} vendedores")
df_kpi_ventas.show(5)

In [0]:
# ── KPI 4: EFECTIVIDAD DE VISITAS POR VENDEDOR ───────────────
df_kpi_visitas = df_visitas \
    .groupBy("id_vendedor") \
    .agg(
        count("id_visita").alias("total_visitas"),
        sum("visita_efectiva").alias("visitas_con_acuerdo"),
        sum(when(col("tipo_visita") == "Prospección",  1).otherwise(0)).alias("prospeccion"),
        sum(when(col("tipo_visita") == "Seguimiento",  1).otherwise(0)).alias("seguimiento"),
        sum(when(col("tipo_visita") == "Cobro",        1).otherwise(0)).alias("cobro"),
        round(sum("valor_cartera"), 2).alias("cartera_gestionada")
    ) \
    .withColumn("pct_efectividad",
        round((col("visitas_con_acuerdo") / col("total_visitas")) * 100, 2)
    ) \
    .withColumn("semaforo_efectividad",
        when(col("pct_efectividad") >= 50, "VERDE")
       .when(col("pct_efectividad") >= 30, "AMARILLO")
       .otherwise("ROJO")
    ) \
    .withColumn("_updated_at", current_timestamp())

print(f"✅ KPI Visitas: {df_kpi_visitas.count()} vendedores")
df_kpi_visitas.show(5)

In [0]:
# ── GUARDAR KPIs EN CURRENT ───────────────────────────────────
kpis = {
    CURRENT_KPI_ENTREGAS:   df_kpi_entregas,
    CURRENT_KPI_INVENTARIO: df_kpi_inventario,
    CURRENT_KPI_VENTAS:     df_kpi_ventas,
    CURRENT_KPI_VISITAS:    df_kpi_visitas,
}

for ruta, df in kpis.items():
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .save(ruta)
    nombre = ruta.split("/")[-1]
    print(f"✅ Guardado: {nombre}")